In [4]:
# ==============================================================================
# APA THESAURUS INGESTION: STEP 1 (MANUAL EXTRACT MAPPING)
# 
# ARCHITECTURE NOTE: The APA Thesaurus is closed-access. This script takes a 
# manually compiled CSV (Strategy A - Local Parsing) and standardizes it.
#
# SSSOM ALIGNMENT STRATEGY: 
# 1. Hierarchy: Built bottom-up using the supplied Parent_IDs. Out-of-scope 
#    parents are resolved via a hardcoded dictionary or retained as ID strings.
# 2. CURIEs: Synthesized using the APA prefix and the Concept_ID.
# ==============================================================================

import os
import sys
import pandas as pd
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
external_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "external"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Check PYTHONPATH.")

# --- 2. Registry Lookup & Setup ---
SOURCE_PREFIX = "APA"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="APA Thesaurus of Psychological Index Terms",
    fallback_uri="https://psycnet.apa.org/thesaurus/item?code="
)
SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']

input_csv_file = os.path.join(external_data_dir, "APA.csv")
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

if not os.path.exists(input_csv_file):
    print(f"[!] Critical Error: Could not find manual extract at {input_csv_file}")
    sys.exit(1)

# --- HARDCODED PARENTS FOR OUT-OF-SCOPE ROOTS ---
# Fill this dictionary with the results from the QA Report!
HARDCODED_PARENTS = {
    '04500' : 'Attitudes',
    '16000' : 'Education',
    '16900' : 'Emotional States',
    '19022' : 'Facilities',
    '22396' : 'Health Care Services',
    '23535' : 'Humanities',
    '35780' : 'Organizations',
    '37980' : 'Personnel',
    '38270' : 'Philosophies',
    '40070' : 'Prejudice',
    '42110' : 'Psychotherapy',
    '45790' : 'Schools',
    '48250' : 'Social Influences',
    '81770' : 'Tests and Measures'
}

# --- 3. Load Data & Clean Headers ---
print(f"[*] Loading manual extract from {os.path.basename(input_csv_file)}...")
df_input = pd.read_csv(input_csv_file, dtype=str, sep=None, engine='python', encoding='utf-8-sig')

# Strip invisible whitespace from all column headers
df_input.columns = df_input.columns.str.strip()

expected_cols = ['Primary_Label', 'Concept_ID', 'Parent_IDs', 'Synonyms', 'Description']
for col in expected_cols:
    if col not in df_input.columns:
        df_input[col] = ""

df_input.fillna("", inplace=True)

# --- 4. QA Reports ---
print("\n--- QA REPORT: DUPLICATE CONCEPT IDs ---")
duplicate_ids = df_input[df_input.duplicated(subset=['Concept_ID'], keep=False)]
if not duplicate_ids.empty:
    print(f"[!] WARNING: Found {len(duplicate_ids)} rows with duplicate Concept_IDs!")
    print(duplicate_ids[['Concept_ID', 'Primary_Label']].sort_values(by='Concept_ID').to_string(index=False))
else:
    print("[✓] Passed: No duplicate Concept IDs found.")

print("\n--- QA REPORT: UNRESOLVED PARENT IDs ---")
# Extract all unique Concept_IDs
known_concept_ids = set(df_input['Concept_ID'].str.strip().tolist())

# Extract all unique Parent_IDs (handling potential pipe separation)
all_parent_ids = set()
for p_ids_str in df_input['Parent_IDs']:
    for p_id in p_ids_str.split('|'):
        clean_p_id = p_id.strip()
        if clean_p_id:
            all_parent_ids.add(clean_p_id)

# Find parents not in our concept list and not already hardcoded
missing_parent_ids = all_parent_ids - known_concept_ids - set(HARDCODED_PARENTS.keys())

if missing_parent_ids:
    print(f"[!] Found {len(missing_parent_ids)} Parent_IDs outside the extracted dataset.")
    print("Click the URIs below to find their labels, then add them to the HARDCODED_PARENTS dict:\n")
    for missing_id in sorted(missing_parent_ids):
        print(f" - {missing_id}: {BASE_URI}{missing_id}")
else:
    print("[✓] Passed: All Parent_IDs resolve to a known concept or hardcoded label.")
print("----------------------------------------\n")

# Drop exact duplicates
df_input = df_input.drop_duplicates(subset=['Concept_ID'], keep='first')

# --- 5. Prepare Hierarchy Mappings ---
id_to_label = dict(zip(df_input['Concept_ID'].str.strip(), df_input['Primary_Label'].str.strip()))
id_to_parent = dict(zip(df_input['Concept_ID'].str.strip(), df_input['Parent_IDs'].str.strip()))

def build_hierarchy_path(cid, visited=None):
    if visited is None:
        visited = set()
        
    if cid in visited:
        return "[Cycle Detected]" 
    visited.add(cid)
    
    # 1. Resolve current label
    if cid in id_to_label:
        label = id_to_label[cid]
    elif cid in HARDCODED_PARENTS:
        label = HARDCODED_PARENTS[cid]
    else:
        label = str(cid)
        
    # 2. Check for parents
    parent_id = id_to_parent.get(cid, "")
    if not parent_id:
        return label
        
    primary_parent = parent_id.split('|')[0].strip()
    
    # 3. Recurse upward
    if primary_parent:
        parent_path = build_hierarchy_path(primary_parent, visited.copy())
        return f"{parent_path} > {label}"
        
    return label

# --- 6. Extraction & Formatting ---
print("[*] Mapping data to Bronze Layer schema...")
if os.path.exists(raw_ingest_file):
    os.remove(raw_ingest_file)

total_processed = 0

for _, row in df_input.iterrows():
    cid = row['Concept_ID'].strip()
    if not cid: continue
        
    extracted_data = {
        'Source_System': SOURCE_NAME,
        'Primary_Label': row['Primary_Label'].strip(),
        'CURIE': f"{SOURCE_PREFIX}:{cid}",
        'Formal_Label': "", 
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': build_hierarchy_path(cid),
        'Synonyms': row['Synonyms'].strip(),
        'Description': row['Description'].strip(),
        'Parent_IDs': row['Parent_IDs'].strip(),
        'Concept_ID': cid,
        'URI': f"{BASE_URI}{cid}",
        'Has_Translation': "",
        'Status': "active",
        'Crosswalks': ""
    }
    
    clean_row = finalize_row(extracted_data)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )
    total_processed += 1

print(f"\n[COMPLETE] Successfully mapped {total_processed} APA concepts to the standard schema.")

[*] Loading manual extract from APA.csv...

--- QA REPORT: DUPLICATE CONCEPT IDs ---
[✓] Passed: No duplicate Concept IDs found.

--- QA REPORT: UNRESOLVED PARENT IDs ---
[✓] Passed: All Parent_IDs resolve to a known concept or hardcoded label.
----------------------------------------

[*] Mapping data to Bronze Layer schema...

[COMPLETE] Successfully mapped 82 APA concepts to the standard schema.
